# 01 · 환경과 AI Core 연결

**목표**: Jupyter kernel에서 AI Core 연결을 확인하고, 이후 실습에 사용할 RUNNING deployment를 찾습니다.

AI Core service key는 SDK가 읽는 환경변수로 전달합니다. 비밀값은 루트 .env에만 두고 Git에 올리지 않습니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 설정과 deployment 확인 (UI)</h2>
<p style="margin: 7px 0;">아래 UI에서 AI Core 연결 값을 입력하거나 기존 값을 확인하세요. 빈 입력은 기존 .env 값을 유지하며 client secret은 다시 표시하지 않습니다.</p>
<ol style="margin: 7px 0 0; padding-left: 1.3em;">
<li>필요한 값을 입력하고 Save local settings를 누릅니다.</li>
<li>Check RUNNING deployments를 눌러 foundation-models와 orchestration 배포를 확인합니다.</li>
<li>정상이라면 다음 LLM 노트북으로 진행합니다.</li>
</ol>
  </div>
</div>


In [15]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / '1_gen_ai_hub' / 'src').exists())
sys.path.insert(0, str(project_root / '1_gen_ai_hub' / 'src'))
from configuration_ui import build_configuration_ui
configuration_ui = build_configuration_ui()
configuration_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", width="100%", height=900)

* Running on local URL:  http://127.0.0.1:7877
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>설정 UI에서 AI Core 설정 누락이 없다고 표시된다.</li>
<li>Deployment status에 RUNNING deployment가 하나 이상 보인다.</li>
<li>이후 노트북에서도 같은 루트 .env와 Jupyter kernel을 사용한다.</li>
</ul>
<p style="margin: 7px 0;">문제가 없으면 다음으로 진행하세요. 아래 코드는 UI가 수행한 작업을 Python SDK로 확인해 보고 싶은 경우에만 실행합니다.</p>
  </div>
</div>


<hr style="border: 0; border-top: 2px solid #c7d2e1; margin: 32px 0 14px;">
<div style="margin: 0 0 18px; color: #4b5f78; font-size: .86em; font-weight: 700; letter-spacing: .04em;">🧩 아래부터 코드 베이스 실습</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">이 아래 셀들은 UI의 동작을 Python 코드로 확인하는 심화 자료입니다. 기본 실습에는 실행할 필요가 없습니다.</p>
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">설정 상태 읽기</h3>
<p style="margin: 7px 0;">비밀값 자체는 출력하지 않고, 누락된 AI Core 설정 이름만 확인합니다.</p>
  </div>
</div>


In [16]:
# UI와 같은 dotenv 파서로 .env 값을 현재 Jupyter kernel 환경에 적용합니다.
import os
from pathlib import Path
from configuration_ui import read_environment_values

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "1_gen_ai_hub").is_dir())
values = read_environment_values(project_root / ".env")
os.environ.update({key: value for key, value in values.items() if value})

required = ["AICORE_BASE_URL", "AICORE_AUTH_URL", "AICORE_CLIENT_ID", "AICORE_CLIENT_SECRET"]
missing = [key for key in required if not os.getenv(key)]
settings = {
    "base_url": os.environ.get("AICORE_BASE_URL", "").rstrip("/"),
    "auth_url": os.environ.get("AICORE_AUTH_URL", "").rstrip("/"),
    "client_id": os.environ.get("AICORE_CLIENT_ID", ""),
    "client_secret": os.environ.get("AICORE_CLIENT_SECRET", ""),
    "resource_group": os.environ.get("AICORE_RESOURCE_GROUP", "default"),
}
print("LLM model:", os.getenv("LLM_MODEL", "(not set)"))
print("AI Core missing fields:", missing)


LLM model: gpt-5.6-luna
AI Core missing fields: []


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">RUNNING deployment 조회 코드</h3>
<p style="margin: 7px 0;">AI Core REST API로 RUNNING deployment를 조회합니다. 결과에는 모델명, scenario, 상태를 표시합니다. Orchestration처럼 모델명이 없는 서비스 deployment는 scenarioId로 구분합니다.</p>
  </div>
</div>


In [17]:
# OAuth token을 받은 뒤 AI Core REST API에서 RUNNING deployment만 고릅니다.
# token과 client secret은 절대 출력하지 않습니다.
import requests

token_response = requests.post(
    f"{settings['auth_url']}/oauth/token",
    data={"grant_type": "client_credentials"},
    auth=(settings["client_id"], settings["client_secret"]),
    timeout=20,
)
token_response.raise_for_status()
access_token = token_response.json()["access_token"]

deployment_response = requests.get(
    f"{settings['base_url']}/v2/lm/deployments?$top=200",
    headers={"Authorization": f"Bearer {access_token}", "AI-Resource-Group": settings["resource_group"]},
    timeout=20,
)
deployment_response.raise_for_status()
deployments = [item for item in deployment_response.json().get("resources", []) if item.get("status") == "RUNNING"]
def model_name(deployment):
    return (
        deployment.get("details", {}).get("resources", {})
        .get("backend_details", {}).get("model", {}).get("name", "")
    )

[
    {"model_name": model_name(item) or "(model name unavailable)",
     "scenario_id": item.get("scenarioId", "unknown"),
     "status": item.get("status", "unknown")}
    for item in deployments
]


[{'model_name': 'gpt-realtime',
  'scenario_id': 'foundation-models',
  'status': 'RUNNING'},
 {'model_name': 'amazon--titan-embed-image',
  'scenario_id': 'foundation-models',
  'status': 'RUNNING'},
 {'model_name': 'text-embedding-3-large',
  'scenario_id': 'foundation-models',
  'status': 'RUNNING'},
 {'model_name': 'gemini-3.1-flash-image',
  'scenario_id': 'foundation-models',
  'status': 'RUNNING'},
 {'model_name': 'gpt-realtime',
  'scenario_id': 'foundation-models',
  'status': 'RUNNING'},
 {'model_name': 'cohere-rerank-pro',
  'scenario_id': 'foundation-models',
  'status': 'RUNNING'},
 {'model_name': 'anthropic--claude-4.8-opus',
  'scenario_id': 'foundation-models',
  'status': 'RUNNING'},
 {'model_name': 'anthropic--claude-4.6-sonnet',
  'scenario_id': 'foundation-models',
  'status': 'RUNNING'},
 {'model_name': 'anthropic--claude-4.8-opus',
  'scenario_id': 'foundation-models',
  'status': 'RUNNING'},
 {'model_name': 'gpt-5.6-terra',
  'scenario_id': 'foundation-models',
 

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  코드 베이스 실습</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">OBJECT Store · 설정과 연결 확인</h3>
<p style="margin: 7px 0;">S3 호환 Object Storage 설정을 읽고 bucket 접근 권한만 확인합니다.</p>
  </div>
</div>


In [18]:
# OBJECT Store 설정을 읽고 bucket에 쓰지 않는 HeadBucket 요청만 수행합니다.
services_src = project_root / "3_gen_ai_hub_services" / "src"
if str(services_src) not in sys.path:
    sys.path.insert(0, str(services_src))

from grounding.object_storage import ObjectStorageConfig

storage = ObjectStorageConfig.from_environment()
storage.require_configured()
storage.client().head_bucket(Bucket=storage.bucket)
{"status": "ok", "bucket": storage.bucket, "prefix": storage.prefix}


{'status': 'ok',
 'bucket': 'hcp-3943d145-8abb-4d1b-8deb-c035e64abd1c',
 'prefix': 'grounding'}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  코드 베이스 실습</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">Grounding · 설정과 연결 확인</h3>
<p style="margin: 7px 0;">Repository ID를 사용해 등록된 문서 목록만 조회합니다.</p>
  </div>
</div>


In [19]:
# Grounding repository의 문서 목록을 읽기 전용으로 조회합니다.
from lab_common.config import load_settings
from lab_common.grounding_repository import list_documents

repository_id = os.getenv("GROUNDING_REPOSITORY_ID", "").strip()
if not repository_id:
    raise RuntimeError("GROUNDING_REPOSITORY_ID를 .env에 설정하세요.")

documents = list_documents(load_settings(project_root / ".env"), repository_id)
{"status": "ok", "document_count": len(documents)}


{'status': 'ok', 'document_count': 1}

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #f9ab00; background: #fef7e0; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #b06000; letter-spacing: .02em;">📘  코드 베이스 실습 · 선택</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">HANA DB · 설정과 연결 확인</h3>
<p style="margin: 7px 0;">HANA_ENABLED=true일 때만 현재 schema를 읽기 전용으로 조회합니다.</p>
  </div>
</div>


In [20]:
# HANA는 선택 기능입니다. 활성화했을 때만 연결하고 DML/DDL은 수행하지 않습니다.
hana_enabled = os.getenv("HANA_ENABLED", "false").strip().lower() in {"1", "true", "yes", "on"}
if not hana_enabled:
    print("HANA DB: 선택 기능 비활성")
else:
    from hdbcli import dbapi

    required = ["HANA_HOST", "HANA_PORT", "HANA_USER", "HANA_PASSWORD", "HANA_ENCRYPT", "HANA_SSL_VALIDATE_CERTIFICATE"]
    missing = [key for key in required if not os.getenv(key, "").strip()]
    if missing:
        raise RuntimeError("HANA 설정 누락: " + ", ".join(missing))

    connection = dbapi.connect(
        address=os.environ["HANA_HOST"],
        port=int(os.environ["HANA_PORT"]),
        user=os.environ["HANA_USER"],
        password=os.environ["HANA_PASSWORD"],
        encrypt=os.environ["HANA_ENCRYPT"].strip().lower() in {"1", "true", "yes", "on"},
        sslValidateCertificate=os.environ["HANA_SSL_VALIDATE_CERTIFICATE"].strip().lower() in {"1", "true", "yes", "on"},
    )
    try:
        cursor = connection.cursor()
        cursor.execute("SELECT CURRENT_SCHEMA FROM DUMMY")
        schema = cursor.fetchone()[0]
    finally:
        connection.close()

    {"status": "ok", "schema": schema}


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #f9ab00; background: #fef7e0; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #b06000; letter-spacing: .02em;">📘  코드 베이스 실습 · 선택</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h3 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">Graph API · 설정 검증</h3>
<p style="margin: 7px 0;">메일 발송이나 OAuth 갱신 없이 Graph/EAI 설정 형식만 검증합니다.</p>
  </div>
</div>


In [21]:
# Graph API는 선택 기능이며 이 셀은 외부 OAuth 또는 메일 API를 호출하지 않습니다.
from urllib.parse import urlparse

graph_keys = ["GRAPH_TENANT_ID", "GRAPH_CLIENT_ID", "GRAPH_OAUTH_REFRESH_TOKEN", "GRAPH_FORWARDER_B_MAILBOX", "OUTLOOK_GRAPH_SEND_ENDPOINT", "A2A_GRAPH_MAIL_SERVICE_TOKEN", "OUTLOOK_GRAPH_SEND_TIMEOUT_SECONDS", "ALLOW_WRITE_TOOLS"]
configured = any(os.getenv(key, "").strip() for key in graph_keys)
if not configured:
    print("Graph API: 선택 기능 미설정")
else:
    missing = [key for key in graph_keys if not os.getenv(key, "").strip()]
    if missing:
        raise RuntimeError("Graph API 설정 누락: " + ", ".join(missing))

    endpoint = urlparse(os.environ["OUTLOOK_GRAPH_SEND_ENDPOINT"])
    if endpoint.scheme not in {"http", "https"} or not endpoint.netloc:
        raise ValueError("OUTLOOK_GRAPH_SEND_ENDPOINT는 http(s) URL이어야 합니다.")
    if "@" not in os.environ["GRAPH_FORWARDER_B_MAILBOX"]:
        raise ValueError("GRAPH_FORWARDER_B_MAILBOX 이메일 형식을 확인하세요.")
    if float(os.environ["OUTLOOK_GRAPH_SEND_TIMEOUT_SECONDS"]) <= 0:
        raise ValueError("OUTLOOK_GRAPH_SEND_TIMEOUT_SECONDS는 0보다 커야 합니다.")
    if os.environ["ALLOW_WRITE_TOOLS"].strip().lower() not in {"0", "1", "false", "true", "no", "yes", "off", "on"}:
        raise ValueError("ALLOW_WRITE_TOOLS는 boolean이어야 합니다.")

{"status": "ok", "external_call": False}


{'status': 'ok', 'external_call': False}